# Notebook 1 — Data Collection (Web Scraping)

Instructions: Run cells top to bottom. Each section saves a CSV to `data/raw/`.

**The scraped data:** top 10 UEFA-ranked nations (2025/26 country coefficients):
England, Italy, Spain, Germany, France, Netherlands, Portugal, Belgium, Turkey, Czechia.

**Datasets (seasons 2006/07 — 2025/26):**
1. League club data — Transfermarkt `startseite` (one page per league-season gives
   club, squad size, avg age, foreigner count, avg value, total squad value which is all I need for this analysis)
2. Italy tournament squads + market values — Transfermarkt `kader` + manual DNQ playoffs from Wikipedia.
3. Italy international results — manual (but automated) as there are no reliable websites which allow for scraping.

All scrapers save incrementally and are resume-safe: re-running a cell only fetches missing country-season combos.

## 0. Imports and setup

In [2]:
import sys, importlib, time, re, unicodedata, difflib
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))
import helpers
importlib.reload(helpers)          # always pick up latest helpers.py edits
from helpers import parse_market_value, TM_HEADERS, LEAGUE_CODES, LEAGUE_SLUGS

RAW_DIR = Path('../data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

def safe_get(url, headers=TM_HEADERS, sleep=3, retries=3):
    """GET with retry + polite delay; returns None if all attempts fail."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=15)
            time.sleep(sleep)
            if r.status_code == 200:
                return r
            print(f'  Status {r.status_code} (attempt {attempt+1})')
            time.sleep(sleep * 2)
        except Exception as e:
            print(f'  Error: {e} (attempt {attempt+1})')
            time.sleep(sleep * 2)
    return None

print(f'Setup complete. {len(LEAGUE_CODES)} leagues: {list(LEAGUE_CODES.keys())}')

Setup complete. 10 leagues: ['England', 'Italy', 'Spain', 'Germany', 'France', 'Netherlands', 'Portugal', 'Belgium', 'Turkey', 'Czechia']


---
## Diagnostic — sanity check one Transfermarkt page

Important: Run this BEFORE the big scrape. It fetches one Premier League season and reports
exactly what's coming back: HTTP status, response length, whether the `items` table
is found, and what the first row's cells look like. If this fails, the full scrape
will fail too. Often the website tries to block the scraping, I added a 3 second wait time to help with this.

In [3]:
# Single-URL probe: run this before the big loops to confirm TM is reachable and parseable.
test_url = ('https://www.transfermarkt.co.uk/premier-league/startseite/'
            'wettbewerb/GB1/plus/?saison_id=2024')
print(f'GET {test_url}')
r = requests.get(test_url, headers=TM_HEADERS, timeout=15)
print(f'  status      : {r.status_code}')
print(f'  length      : {len(r.text):,} chars')
print(f'  final URL   : {r.url}')

lower = r.text.lower()
for marker in ('cf-chl', 'cloudflare', 'just a moment', 'captcha',
               'access denied', 'verify you are human'):
    if marker in lower:
        print(f'  ! BOT WALL DETECTED — page contains "{marker}"')
        break

soup  = BeautifulSoup(r.text, 'lxml')
table = soup.find('table', {'class': 'items'})
print(f'  items table : {table is not None}')
if table:
    rows = table.find('tbody').find_all('tr') if table.find('tbody') else []
    print(f'  tbody rows  : {len(rows)}')
    for i, row in enumerate(rows[:2]):
        cells = row.find_all('td')
        print(f'\n  -- Sample row {i}: {len(cells)} cells --')
        for j, td in enumerate(cells):
            cls = td.get('class', [])
            txt = td.get_text(strip=True)[:50]
            if cls or txt:
                print(f'    cell {j:2}  classes={cls!s:<40} text={txt!r}')
else:
    print('  -> Page returned without the expected table.')
    print('  -> First 500 chars of body:')
    print('    ', r.text[:500].replace(chr(10), ' '))

GET https://www.transfermarkt.co.uk/premier-league/startseite/wettbewerb/GB1/plus/?saison_id=2024
  status      : 200
  length      : 194,546 chars
  final URL   : https://www.transfermarkt.co.uk/premier-league/startseite/wettbewerb/GB1/plus/?saison_id=2024
  items table : True
  tbody rows  : 20

  -- Sample row 0: 7 cells --
    cell  0  classes=['zentriert', 'no-border-rechts']        text=''
    cell  1  classes=['hauptlink', 'no-border-links']         text='Manchester City'
    cell  2  classes=['zentriert']                            text='44'
    cell  3  classes=['zentriert']                            text='25.6'
    cell  4  classes=['zentriert']                            text='27'
    cell  5  classes=['rechts']                               text='€30.86m'
    cell  6  classes=['rechts']                               text='€1.36bn'

  -- Sample row 1: 7 cells --
    cell  0  classes=['zentriert', 'no-border-rechts']        text=''
    cell  1  classes=['hauptlink', 'no-bord

---
## Dataset 1 — League Club Data (Transfermarkt)

For each of the top 10 UEFA nations, I scrape the league overview (`startseite`) page on
Transfermarkt for every season 2006–2025. One request per league-season yields **every
piece of league-level information we need**:

- `squad_size` — total players
- `avg_age`
- `foreign_players` — count of foreigners per club (league % computed in cleaning)
- `avg_market_value_m` — average per player
- `squad_market_value_m` — total squad value

Scraping only this page (instead of `startseite` + `gastarbeiter` separately, which was the initial way I used) halves the
requests and keeps all variables aligned on the same snapshot. Furthermore recently gasterbeiter has been doubling down on bot protection, this makes it almost impossible to scrape the data. 

The code saves incrementally after each country so progress is not lost if interrupted.

This process will last around 1 hour, the rest of the scraping lasts less than 10 minutes.

In [4]:
SEASONS     = list(range(2006, 2026))
OUTPUT_FILE = RAW_DIR / 'league_club_data.csv'

# Resume: load already-scraped rows so we never re-fetch them
if OUTPUT_FILE.exists():
    existing  = pd.read_csv(OUTPUT_FILE)
    all_rows  = existing.to_dict('records')
    done_keys = set(zip(existing['country'], existing['season_start']))
    print(f'Resuming — {len(done_keys)} country-season rows already scraped')
else:
    all_rows  = []
    done_keys = set()
    print('Starting fresh')

remaining = sum(
    1 for c in LEAGUE_CODES for s in SEASONS
    if (c, s) not in done_keys
)
print(f'{remaining} country-season combos left to scrape')

Resuming — 197 country-season rows already scraped
3 country-season combos left to scrape


In [5]:
def _to_int(s):
    try:    return int(str(s).replace(',', '').strip())
    except ValueError: return None

def _to_float(s):
    try:    return float(str(s).replace(',', '.').strip())
    except ValueError: return None


def extract_club_rows(soup, country, code, season_start):
    """
    Extract all league columns from a TM startseite page.

    Row layout (verified via manual inspection of many pages across seasons and leagues) is roughly:
      cell 0  logo         class='zentriert no-border-rechts'
      cell 1  club name    class='hauptlink no-border-links'
      cell 2  squad size   class='zentriert'          -> int
      cell 3  avg age      class='zentriert'          -> float
      cell 4  foreigners   class='zentriert'          -> int
      cell 5  avg value    class='rechts'             -> float (m)
      cell 6  TOTAL value  class='rechts'             -> float (m)

    I identify cells by class rather than raw index so minor layout shifts
    (an extra logo cell, shuffled columns, etc.) don't silently corrupt the parse.
    """
    results = []
    table   = soup.find('table', {'class': 'items'})
    if not table:
        return results
    tbody = table.find('tbody')
    if not tbody:
        return results

    for row in tbody.find_all('tr'):
        cells = row.find_all('td')
        if len(cells) < 5:
            continue   

        # Club name — td with class 'hauptlink'
        club_td = row.find('td', {'class': lambda c: c and 'hauptlink' in c})
        if not club_td:
            continue
        a         = club_td.find('a')
        club_name = (a.get_text(strip=True) if a else club_td.get_text(strip=True)).strip()
        if not club_name:
            continue

        # 'zentriert' data cells — skip the logo cell with 'no-border-rechts'
        zent = [
            td for td in cells
            if 'zentriert' in td.get('class', [])
            and 'no-border-rechts' not in td.get('class', [])
        ]
        squad_size = _to_int(zent[0].get_text(strip=True))   if len(zent) >= 1 else None
        avg_age    = _to_float(zent[1].get_text(strip=True)) if len(zent) >= 2 else None
        foreigners = _to_int(zent[2].get_text(strip=True))   if len(zent) >= 3 else None

        # 'rechts' money cells — first is avg/player, last is TOTAL squad value
        rechts = [td for td in cells if 'rechts' in td.get('class', [])]
        avg_value_m   = parse_market_value(rechts[0].get_text(strip=True))  if rechts else None
        total_value_m = parse_market_value(rechts[-1].get_text(strip=True)) if rechts else None

        results.append({
            'country':              country,
            'league_code':          code,
            'season':               f'{season_start}-{season_start+1}',
            'season_start':         season_start,
            'club':                 club_name,
            'squad_size':           squad_size,
            'avg_age':              avg_age,
            'foreign_players':      foreigners,
            'avg_market_value_m':   avg_value_m,
            'squad_market_value_m': total_value_m,
        })
    return results


for country, code in LEAGUE_CODES.items():
    slug = LEAGUE_SLUGS[country]
    print(f'\n{country} ({code})')
    league_rows = []

    for season_start in SEASONS:
        if (country, season_start) in done_keys:
            continue
        url  = (
            f'https://www.transfermarkt.co.uk/{slug}/startseite/'
            f'wettbewerb/{code}/plus/?saison_id={season_start}'
        )
        resp = safe_get(url, sleep=3)
        if resp is None:
            print(f'  x {season_start}: failed')
            continue

        clubs = extract_club_rows(BeautifulSoup(resp.text, 'lxml'), country, code, season_start)
        if clubs:
            league_rows.extend(clubs)
            print(f'  ok {season_start}: {len(clubs)} clubs')
        else:
            print(f'  -- {season_start}: no data')

    all_rows.extend(league_rows)
    pd.DataFrame(all_rows).to_csv(OUTPUT_FILE, index=False)
    print(f'  Saved {len(league_rows)} rows for {country}')

league_df = pd.DataFrame(all_rows)
print(f'\nDataset 1 complete — {len(league_df)} rows')


England (GB1)
  ok 2022: 20 clubs
  Saved 20 rows for England

Italy (IT1)
  ok 2010: 20 clubs
  Saved 20 rows for Italy

Spain (ES1)
  Saved 0 rows for Spain

Germany (L1)
  Saved 0 rows for Germany

France (FR1)
  ok 2009: 20 clubs
  Saved 20 rows for France

Netherlands (NL1)
  Saved 0 rows for Netherlands

Portugal (PO1)
  Saved 0 rows for Portugal

Belgium (BE1)
  Saved 0 rows for Belgium

Turkey (TR1)
  Saved 0 rows for Turkey

Czechia (TS1)
  Saved 0 rows for Czechia

Dataset 1 complete — 3681 rows


In [6]:
# Sense check: coverage per country and non-null rates for each metric. This also confirms that the parsing logic is correctly identifying the key columns across the various page layouts, since I get plausible non-null counts for each metric.
print('Rows per country:')
print(league_df.groupby('country').size().sort_values(ascending=False).to_string())

print('\nNon-null counts per column:')
print(league_df[['squad_size','avg_age','foreign_players','avg_market_value_m','squad_market_value_m']]
      .notna().sum().to_string())

print('\nSample — Italy 2023/24:')
print(league_df[(league_df['country']=='Italy') & (league_df['season_start']==2023)]
      .to_string(index=False))

Rows per country:
country
England        400
Italy          400
Spain          400
France         394
Turkey         369
Germany        360
Netherlands    360
Portugal       344
Belgium        332
Czechia        322

Non-null counts per column:
squad_size              3681
avg_age                 3681
foreign_players         3681
avg_market_value_m      3681
squad_market_value_m    3681

Sample — Italy 2023/24:
country league_code    season  season_start                club  squad_size  avg_age  foreign_players  avg_market_value_m  squad_market_value_m
  Italy         IT1 2023-2024          2023         Inter Milan          35     27.1               24               18.71                654.85
  Italy         IT1 2023-2024          2023            AC Milan          42     24.6               26               13.61                571.41
  Italy         IT1 2023-2024          2023          SSC Napoli          35     26.5               22               14.71                515.00
  Italy  

---
## Dataset 2 — Italy Tournament Squads + Market Values (Transfermarkt)

For each major tournament season we scrape the **Italy national team kader page**
on Transfermarkt, which lists every called-up player with their position, club,
and market value — all in one request per season.

For **2018 & 2022** (Italy did not qualify) we manually enter the playoff squad.
Market values for those two squads are looked up from the same TM kader page
for the relevant season.

Columns saved: `year`, `tournament`, `player`, `position`, `club`, `market_value_m`

In [7]:
OUTPUT_FILE_SQ = RAW_DIR / 'italy_squad_selections.csv'

# Map each tournament year to the TM saison_id to fetch.
# Summer tournaments: use the season that just ended (year - 1).
# Autumn/spring playoffs: use the season in progress.
TOURNAMENT_SEASONS = [
    (2006, 'World Cup', 2005),
    (2008, 'Euro',      2007),
    (2010, 'World Cup', 2009),
    (2012, 'Euro',      2011),
    (2014, 'World Cup', 2013),
    (2016, 'Euro',      2015),
    (2021, 'Euro',      2020),
    (2024, 'Euro',      2023),
]


def scrape_tm_italy_squad(year, tournament, season):
    """Scrape the Italy national team kader page for a given TM season.

    Note: the `club` column on this page is not reliably extracted — the
    tournament-squad layout hides the club affiliation behind a tooltip that
    has changed selectors across TM redesigns. For this project we only need
    the player name and market value, so leaving `club` as a blank column is
    acceptable — the downstream cleaning/modelling never reads it.
    """
    url  = (f'https://www.transfermarkt.co.uk/italien/kader/verein/3376'
            f'/saison_id/{season}/plus/1')
    resp = safe_get(url, sleep=3)
    if resp is None:
        print(f'  x {year} {tournament}: request failed')
        return []

    soup  = BeautifulSoup(resp.text, 'lxml')
    table = soup.find('table', {'class': 'items'})
    if not table:
        print(f'  x {year} {tournament}: no table found')
        return []

    rows = []
    for tr in table.find('tbody').find_all('tr'):
        if 'class' in tr.attrs and 'spacer' in tr['class']:
            continue

        # Player name
        name_td = tr.find('td', {'class': lambda c: c and 'hauptlink' in c})
        if not name_td:
            continue
        name_a = name_td.find('a')
        if not name_a:
            continue
        player = (name_a.get('title') or name_a.get_text(strip=True)).strip()
        if not player:
            continue

        # Position
        pos_td   = tr.find('td', {'class': lambda c: c and 'posrela' in c})
        position = ''
        if pos_td:
            inner = pos_td.find('table')
            if inner:
                position = inner.get_text(strip=True)
        if not position:
            for td in tr.find_all('td'):
                txt = td.get_text(strip=True)
                if txt in ('GK', 'CB', 'LB', 'RB', 'LWB', 'RWB', 'DF',
                           'DM', 'CM', 'AM', 'LM', 'RM', 'MF',
                           'LW', 'RW', 'SS', 'CF', 'FW', 'ST'):
                    position = txt
                    break

        # Market value: td.rechts.hauptlink first; fall back to last td.rechts if the former is missing (layout changes have moved the value cell around, but it's always the rightmost 'rechts' cell)
        val_td = tr.find('td', {'class': lambda c: c and 'rechts' in c and 'hauptlink' in c})
        if val_td is None:
            rechts_tds = [td for td in tr.find_all('td') if 'rechts' in td.get('class', [])]
            val_td = rechts_tds[-1] if rechts_tds else None
        value  = parse_market_value(val_td.get_text(strip=True)) if val_td else None

        rows.append({
            'year':           year,
            'tournament':     tournament,
            'player':         player,
            'position':       position,
            'market_value_m': value,
        })

    return rows


tm_rows = []
for year, tournament, season in TOURNAMENT_SEASONS:
    rows = scrape_tm_italy_squad(year, tournament, season)
    if rows:
        tm_rows.extend(rows)
        matched = sum(1 for r in rows if r['market_value_m'] is not None)
        print(f'  ok {year} {tournament}: {len(rows)} players, {matched} with values')
    time.sleep(2)

squad_df = pd.DataFrame(tm_rows)
print(f'\nTM scrape done: {len(squad_df)} rows')
squad_df


  ok 2006 World Cup: 88 players, 44 with values
  ok 2008 Euro: 90 players, 45 with values
  ok 2010 World Cup: 94 players, 47 with values
  ok 2012 Euro: 78 players, 39 with values
  ok 2014 World Cup: 94 players, 47 with values
  ok 2016 Euro: 94 players, 47 with values
  ok 2021 Euro: 98 players, 49 with values
  ok 2024 Euro: 98 players, 49 with values

TM scrape done: 734 rows


,year,tournament,player,position,market_value_m
0,2006,World Cup,Angelo Peruzzi,,4.0
1,2006,World Cup,Angelo Peruzzi,,NaN
2,2006,World Cup,Christian Abbiati,,3.0
3,2006,World Cup,Christian Abbiati,,NaN
4,2006,World Cup,Marco Amelia,,3.0
...,...,...,...,...,...
729,2024,Euro,Ciro Immobile,,NaN
730,2024,Euro,Moise Kean,,20.0
731,2024,Euro,Moise Kean,,NaN
732,2024,Euro,Mateo Retegui,,8.0


In [8]:
# Manually entered playoff squads for 2018 and 2022 (Italy did not qualify). And when I say manually entered, I maen used AI to enter them.
# Market values are then looked up from the TM kader page for the relevant season.

def normalize_name(name):
    name = unicodedata.normalize('NFKD', str(name))
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-z\s]', '', name.lower())
    return ' '.join(name.split())

def lookup_value(player, val_dict, threshold=0.75):
    norm = normalize_name(player)
    if norm in val_dict:
        return val_dict[norm]
    matches = difflib.get_close_matches(norm, val_dict.keys(), n=1, cutoff=threshold)
    return val_dict[matches[0]] if matches else None

def tm_kader_values(season):
    """Return {normalized_name: market_value_m} for a TM Italy kader season."""
    url  = (f'https://www.transfermarkt.co.uk/italien/kader/verein/3376'
            f'/saison_id/{season}/plus/1')
    resp = safe_get(url, sleep=3)
    if resp is None:
        return {}
    soup  = BeautifulSoup(resp.text, 'lxml')
    table = soup.find('table', {'class': 'items'})
    if not table:
        return {}
    out = {}
    for tr in table.find('tbody').find_all('tr'):
        name_td = tr.find('td', {'class': lambda c: c and 'hauptlink' in c})
        if not name_td:
            continue
        name_a = name_td.find('a')
        if not name_a:
            continue
        raw = (name_a.get('title') or name_a.get_text(strip=True)).strip()
        vtd = tr.find('td', {'class': lambda c: c and 'rechts' in c and 'hauptlink' in c})
        if vtd is None:
            rechts_tds = [td for td in tr.find_all('td') if 'rechts' in td.get('class', [])]
            vtd = rechts_tds[-1] if rechts_tds else None
        val = parse_market_value(vtd.get_text(strip=True)) if vtd else None
        if raw and val is not None:
            out[normalize_name(raw)] = val
    return out


dnq_squads = [
    # 2018 WC qualifying playoff (vs Sweden, Nov 2017) (watched this match live, so confident in the squad list)
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'GK','player':'Gianluigi Buffon',      'club':'Juventus'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'GK','player':'Salvatore Sirigu',     'club':'Torino'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'GK','player':'Mattia Perin',         'club':'Genoa'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Leonardo Bonucci',     'club':'AC Milan'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Giorgio Chiellini',    'club':'Juventus'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Andrea Barzagli',      'club':'Juventus'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Matteo Darmian',       'club':'Manchester Utd'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Mattia De Sciglio',    'club':'Juventus'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Alessandro Florenzi',  'club':'Roma'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'DF','player':'Davide Astori',        'club':'Fiorentina'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Daniele De Rossi',     'club':'Roma'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Marco Verratti',       'club':'PSG'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Antonio Candreva',     'club':'Inter Milan'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Marco Parolo',         'club':'Lazio'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Federico Bernardeschi','club':'Juventus'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Giacomo Bonaventura',  'club':'AC Milan'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'MF','player':'Lorenzo Pellegrini',   'club':'Roma'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'FW','player':'Ciro Immobile',        'club':'Lazio'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'FW','player':'Lorenzo Insigne',      'club':'Napoli'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'FW','player':'Andrea Belotti',       'club':'Torino'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'FW','player':'Stephan El Shaarawy',  'club':'Roma'},
    {'year':2018,'tournament':'World Cup (Qualifier)','position':'FW','player':'Eder',                 'club':'Inter Milan'},
    # 2022 WC qualifying playoff (vs N. Macedonia, Mar 2022)
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'GK','player':'Gianluigi Donnarumma', 'club':'PSG'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'GK','player':'Salvatore Sirigu',     'club':'Genoa'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'GK','player':'Alex Meret',           'club':'Napoli'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Leonardo Bonucci',     'club':'Juventus'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Giorgio Chiellini',    'club':'Juventus'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Francesco Acerbi',     'club':'Lazio'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Alessandro Florenzi',  'club':'AC Milan'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Giovanni Di Lorenzo',  'club':'Napoli'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Emerson Palmieri',     'club':'Lyon'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'DF','player':'Alessandro Bastoni',   'club':'Inter Milan'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Marco Verratti',       'club':'PSG'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Nicolo Barella',       'club':'Inter Milan'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Jorginho',             'club':'Chelsea'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Manuel Locatelli',     'club':'Juventus'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Lorenzo Pellegrini',   'club':'Roma'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Sandro Tonali',        'club':'AC Milan'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'MF','player':'Bryan Cristante',      'club':'Roma'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Lorenzo Insigne',      'club':'Napoli'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Ciro Immobile',        'club':'Lazio'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Federico Chiesa',      'club':'Juventus'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Giacomo Raspadori',    'club':'Sassuolo'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Joao Pedro',           'club':'Cagliari'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Gianluca Scamacca',    'club':'Sassuolo'},
    {'year':2022,'tournament':'World Cup (Qualifier)','position':'FW','player':'Domenico Berardi',     'club':'Sassuolo'},
]

dnq_values = {
    2018: tm_kader_values(2017),
    2022: tm_kader_values(2021),
}
for row in dnq_squads:
    vdict = dnq_values.get(row['year'], {})
    row['market_value_m'] = lookup_value(row['player'], vdict)

squad_df = pd.concat([squad_df, pd.DataFrame(dnq_squads)], ignore_index=True)
# Sort so rows WITH a market value come first, then deduplicate, this ensures we keep the valued row when one duplicate has it and the other doesn't.
squad_df = (squad_df
    .sort_values('market_value_m', ascending=False, na_position='last')
    .drop_duplicates(subset=['year', 'player'], keep='first')
    .reset_index(drop=True))
squad_df.to_csv(OUTPUT_FILE_SQ, index=False)
print(f'Dataset 2 complete — {len(squad_df)} rows')
print(squad_df.groupby(['year','tournament']).size().to_string())

Dataset 2 complete — 413 rows
year  tournament           
2006  World Cup                44
2008  Euro                     45
2010  World Cup                47
2012  Euro                     39
2014  World Cup                47
2016  Euro                     47
2018  World Cup (Qualifier)    22
2021  Euro                     49
2022  World Cup (Qualifier)    24
2024  Euro                     49


In [9]:
# Sense check (again): squad value coverage and averages per tournament
summary = (squad_df.groupby(['year', 'tournament'])
    .agg(
        total_players  = ('player',         'count'),
        valued_players = ('market_value_m', 'count'),   # non-null only
        avg_value_m    = ('market_value_m', 'mean'),    # NaN-safe mean
    )
    .reset_index()
)
summary['pct_valued'] = (summary['valued_players'] / summary['total_players'] * 100).round(1)

print(summary[['year','tournament','total_players','valued_players','pct_valued','avg_value_m']]
      .to_string(index=False))

low = summary[summary['pct_valued'] < 70]
if len(low):
    print('\nLow coverage (<70%) — avg_value_m unreliable for:')
    print(low[['year','tournament','pct_valued']].to_string(index=False))
else:
    print('\nAll tournaments >= 70% coverage — averages are reliable.')

 year            tournament  total_players  valued_players  pct_valued  avg_value_m
 2006             World Cup             44              42        95.5     9.032143
 2008                  Euro             45              45       100.0     9.368889
 2010             World Cup             47              47       100.0    10.226596
 2012                  Euro             39              39       100.0    12.074359
 2014             World Cup             47              47       100.0    10.804255
 2016                  Euro             47              47       100.0    10.325532
 2018 World Cup (Qualifier)             22              19        86.4    23.868421
 2021                  Euro             49              49       100.0    25.081633
 2022 World Cup (Qualifier)             24              22        91.7    30.227273
 2024                  Euro             49              49       100.0    20.704082

All tournaments >= 70% coverage — averages are reliable.


---
## Dataset 3 — Italy International Results (Manual)

Small dataset of Italy's World Cup and Euro results 1980–2026, this was simply not worth the scrape.

In [10]:
italy_results_raw = [
   
    {'year': 2006, 'tournament': 'World Cup', 'result': 'Winner',          'host': 'Germany'},
    {'year': 2010, 'tournament': 'World Cup', 'result': 'Group Stage',     'host': 'South Africa'},
    {'year': 2014, 'tournament': 'World Cup', 'result': 'Group Stage',     'host': 'Brazil'},
    {'year': 2018, 'tournament': 'World Cup', 'result': 'Did Not Qualify', 'host': 'Russia'},
    {'year': 2022, 'tournament': 'World Cup', 'result': 'Did Not Qualify', 'host': 'Qatar'},
    {'year': 2026, 'tournament': 'World Cup', 'result': 'Qualified',       'host': 'USA/Canada/Mexico'},
    {'year': 2000, 'tournament': 'Euro', 'result': 'Runner-up',       'host': 'Belgium/Netherlands'},
    {'year': 2004, 'tournament': 'Euro', 'result': 'Group Stage',     'host': 'Portugal'},
    {'year': 2008, 'tournament': 'Euro', 'result': 'Quarter-final',   'host': 'Austria/Switzerland'},
    {'year': 2012, 'tournament': 'Euro', 'result': 'Runner-up',       'host': 'Poland/Ukraine'},
    {'year': 2016, 'tournament': 'Euro', 'result': 'Quarter-final',   'host': 'France'},
    {'year': 2021, 'tournament': 'Euro', 'result': 'Winner',          'host': 'Various'},
    {'year': 2024, 'tournament': 'Euro', 'result': 'Round of 16',     'host': 'Germany'},
]
SCORE_MAP = {
    'Winner': 7, 'Runner-up': 6, 'Semi-final': 5, 'Quarter-final': 4,
    'Round of 16': 3, 'Group Stage': 2, 'Did Not Qualify': 1, 'Qualified': 3,
}
results_df = pd.DataFrame(italy_results_raw)
results_df['performance_score'] = results_df['result'].map(SCORE_MAP)
results_df['qualified'] = (results_df['result'] != 'Did Not Qualify').astype(int)
results_df.to_csv(RAW_DIR / 'italy_results.csv', index=False)
print('Dataset 3 complete')
print(results_df[['year','tournament','result','performance_score']].to_string(index=False))

Dataset 3 complete
 year tournament          result  performance_score
 2006  World Cup          Winner                  7
 2010  World Cup     Group Stage                  2
 2014  World Cup     Group Stage                  2
 2018  World Cup Did Not Qualify                  1
 2022  World Cup Did Not Qualify                  1
 2026  World Cup       Qualified                  3
 2000       Euro       Runner-up                  6
 2004       Euro     Group Stage                  2
 2008       Euro   Quarter-final                  4
 2012       Euro       Runner-up                  6
 2016       Euro   Quarter-final                  4
 2021       Euro          Winner                  7
 2024       Euro     Round of 16                  3


---
## Final check

In [11]:
files = [
    'league_club_data.csv',
    'italy_squad_selections.csv',
    'italy_results.csv',
]

all_ok = True
for fname in files:
    fpath = RAW_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f'  ok      {fname:<40} {len(df):>6,} rows')
    else:
        print(f'  MISSING {fname}')
        all_ok = False

if all_ok:
    print('\nAll files present. Move to 02_cleaning.ipynb.')
else:
    print('\nRe-run the missing dataset cells above, then re-check.')

  ok      league_club_data.csv                      3,681 rows
  ok      italy_squad_selections.csv                  413 rows
  ok      italy_results.csv                            13 rows

All files present. Move to 02_cleaning.ipynb.
